# RSNA-2022-CSFD 解压 + 切 npy（Colab）

假设数据已放在 **`dataset/pretrain/RSNA-2022-CSFD/download/`**（例如 Kaggle 竞赛包或其中唯一顶层目录）。本笔记本会：

1. 创建与同目录平级的 **`unzip/`**、**`npy/`**
2. **解压与校验**：先运行 **「解压共用设置」**，再依次 **四个解压格**（同上四个 zip）；完成后可运行 **四个「文件数校验」格**，对照 zip 内文件条目数与 `unzip/` 下已落盘文件数是否一致。**Colab 本地盘很小**：`TMPDIR`/`TMP`/`TEMP` 指到 Drive 下 **`RSNA-2022-CSFD/.colab_tmp`**，**逐文件解压 + 进度条**。
3. 在 **`unzip/`** 有内容时从 `unzip/` 扫描体积；若 `unzip/` 为空而 **`download/`** 里已是解压好的目录，则回退为从 **`download/`** 扫描（无需再拷一遍）
4. **`pip install` 之后**先运行 **「了解目录结构」** 代码格，确认顶层/扩展名/DICOM 叶子目录等；再运行切 npy。

**切 npy 逻辑**与 `colab_abdomenct1k_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` 一致：

- 每个 3D 体积：**50** 个 npy（轴向 **z≥128**）或 **33** 个（**z<128**，`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**输入**：**DICOM** — 每个仅含 `.dcm`、且无子目录的叶子目录视为一个 3D 序列（与你当前解压结构一致：`train_part_*/…/StudyUID/*.dcm`、`test_images/…/StudyUID/*.dcm`）。**NIfTI** — 仅当存在体积 nii 时收集；**自动跳过路径中含 `segmentations/` 的 `.nii`**（竞赛为分割标注，非 CT，不做 patch）。可选 **`RSNA_INCLUDE_TEST_IMAGES=False`** 排除 `test_images`。跳过 **`__MACOSX`**。

**输出**：`.npy` 扁平放在 **`npy/`**；命名形如 `subdir_stem_0_1.npy`（由相对路径与序号组成，避免重名）。

---

### 超大压缩包（例如约 189GB、仅一个 zip）与 Colab 本地盘（约一百～两百多 GB）

- **可以这样做的前提**：`*.zip` 放在 **Google Drive** 的 `download/` 里（路径形如 `/content/drive/MyDrive/...`），解压目标 **`unzip/` 也在同一块 Drive**。这样 **189GB 占用的是 Drive 配额**，**不是** Colab 根分区 `/` 的 200GB；逐文件解压只是把数据**流式写出到 Drive**，不会在本地再完整复制一份 189GB。
- **不行的情况**：若把 zip 放在 **`/content`**、或先 **`!cp` 到本地** 再解压，本地盘会被 **zip 体积 + 其它缓存** 撑满，与「只有 200 多 GB」矛盾。
- **仍可能 `Errno 28` 的原因**：挂载 Drive 的 FUSE **偶发**在本地做缓存；若多次失败，更稳妥的是 **在本机 / 大硬盘服务器上解压**，再同步到 Drive，或换 **Colab Pro / 更大临时盘** 的机型。
- **空间要算两处**：除 Colab 本地外，**解压后的未压缩总体积**常大于 zip，请确认 **Google Drive 剩余空间** 也足够（脚本会打印 zip 体积与磁盘用量）。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

BASE = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD"
DOWNLOAD = os.path.join(BASE, "download")
UNZIP_ROOT = os.path.join(BASE, "unzip")
NPY_ROOT = os.path.join(BASE, "npy")

for p in (BASE, DOWNLOAD, UNZIP_ROOT, NPY_ROOT):
    os.makedirs(p, exist_ok=True)

print("已创建/确认目录:")
print("  ", DOWNLOAD)
print("  ", UNZIP_ROOT)
print("  ", NPY_ROOT)

In [ ]:
# 解压共用设置（须先运行本格，再依次运行下面四个「解压 ①～④」格）
# TMP 指到 Drive；逐文件解压 + 进度；不做 testzip。
!pip install -q tqdm

import os
import shutil
import zipfile

DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/download"
UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"
BASE = os.path.dirname(DOWNLOAD)
TMP_ON_DRIVE = os.path.join(BASE, ".colab_tmp")
os.makedirs(TMP_ON_DRIVE, exist_ok=True)
os.makedirs(UNZIP_ROOT, exist_ok=True)
for _k in ("TMPDIR", "TMP", "TEMP"):
    os.environ[_k] = TMP_ON_DRIVE

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def _print_disk():
    try:
        u = shutil.disk_usage("/")
        print(f"[分区 /] 可用 {u.free / 2**30:.2f} GB / 共 {u.total / 2**30:.2f} GB")
    except Exception:
        pass
    try:
        u2 = shutil.disk_usage(BASE)
        print(f"[Drive 数据目录] 可用 {u2.free / 2**30:.2f} GB（路径: {BASE}）")
    except Exception:
        pass
    print(f"[临时目录 TMPDIR] -> {TMP_ON_DRIVE}")


def _safe_extract_member(zf, member, dest_dir):
    dest_dir = os.path.realpath(dest_dir)
    out_path = os.path.realpath(os.path.join(dest_dir, member.filename))
    try:
        cp = os.path.commonpath([dest_dir, out_path])
    except ValueError:
        raise ValueError(f"非法路径: {member.filename}") from None
    if cp != dest_dir:
        raise ValueError(f"非法路径: {member.filename}")
    return zf.extract(member, dest_dir)


def extract_one_zip_progress(zpath, out_dir, label):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        total_sz = sum(m.file_size for m in members)
        n = len(members)
        print(f"    条目数: {n} 个文件，解压后约 {total_sz / 2**30:.2f} GiB（未压缩大小）")
        it = members
        if tqdm is not None:
            it = tqdm(members, desc=label[:48], unit="file", mininterval=1.0)
        done_b = 0
        for j, m in enumerate(it, start=1):
            _safe_extract_member(zf, m, out_dir)
            done_b += m.file_size
            if tqdm is None and (j == 1 or j % 500 == 0 or j == n):
                pct = 100.0 * done_b / total_sz if total_sz else 0
                print(f"    进度 {j}/{n} 文件 (~{pct:.1f}% 按未压缩字节)")
    return n


def _must_be_on_mounted_drive(path, label="zip"):
    rp = os.path.realpath(path)
    if not rp.startswith("/content/drive"):
        raise RuntimeError(
            f"{label} 必须位于已挂载的 Google Drive 下（路径应以 /content/drive 开头）。\n"
            f"当前: {rp}"
        )


def find_zip_in_download(download_root, basename):
    """在 download 根目录或子目录中按文件名查找 zip。"""
    bn = basename if basename.lower().endswith(".zip") else basename + ".zip"
    direct = os.path.join(download_root, bn)
    if os.path.isfile(direct):
        return direct
    for root, _, files in os.walk(download_root):
        if "__MACOSX" in root.replace("\\", "/"):
            continue
        if bn in files:
            return os.path.join(root, bn)
    raise FileNotFoundError(f"在 {download_root} 下未找到 {bn}")


def run_one_zip(zip_basename, title=None):
    """解压单个 zip 到 UNZIP_ROOT/<文件名去 .zip>/。"""
    bn = zip_basename if zip_basename.lower().endswith(".zip") else zip_basename + ".zip"
    zpath = find_zip_in_download(DOWNLOAD, bn)
    stem = os.path.splitext(os.path.basename(bn))[0]
    out_dir = os.path.join(UNZIP_ROOT, stem)
    rel = os.path.relpath(zpath, DOWNLOAD)
    t = title or bn
    print(f"\n=== {t} ===")
    print(f"    zip: {zpath}")
    print(f"    -> {out_dir}")
    _must_be_on_mounted_drive(zpath, "zip")
    _must_be_on_mounted_drive(out_dir, "解压目标目录")
    zbytes = os.path.getsize(zpath)
    print(f"    zip 文件大小（磁盘）: {zbytes / 2**30:.2f} GiB")
    extract_one_zip_progress(zpath, out_dir, rel)
    print("    OK")
    _print_disk()


def verify_extracted(zip_basename, label=""):
    """校验：zip 内「文件」条目数 vs Drive 上对应解压目录内递归文件数（排除 __MACOSX）。"""
    bn = zip_basename if zip_basename.lower().endswith(".zip") else zip_basename + ".zip"
    zpath = find_zip_in_download(DOWNLOAD, bn)
    stem = os.path.splitext(os.path.basename(bn))[0]
    out_dir = os.path.join(UNZIP_ROOT, stem)
    with zipfile.ZipFile(zpath, "r") as zf:
        n_zip = sum(1 for m in zf.infolist() if not m.is_dir())
    n_disk = 0
    if os.path.isdir(out_dir):
        for dp, _, fs in os.walk(out_dir):
            if "__MACOSX" in dp.replace("\\", "/"):
                continue
            n_disk += len(fs)
    ok = (n_zip == n_disk) and (n_zip > 0)
    tag = label or bn
    print(f"\n=== 文件数校验 {tag} ===")
    print(f"    zip 路径: {zpath}")
    print(f"    解压目录: {out_dir}")
    print(f"    zip 内文件条目数（非目录）: {n_zip}")
    print(f"    Drive 已解压文件数（递归）: {n_disk}")
    if ok:
        print("    结果: 通过（数量一致）")
    else:
        print("    结果: 未通过")
        if not os.path.isdir(out_dir):
            print("    说明: 解压目录不存在，可能尚未解压本包。")
        elif n_disk < n_zip:
            print("    说明: 磁盘文件数偏少，可能解压中断或未完成。")
        else:
            print("    说明: 磁盘文件数偏多，解压目录内可能有额外文件。")
    return ok


print("解压共用函数已就绪：run_one_zip('xxx.zip')；verify_extracted('xxx.zip')")
_print_disk()

### 解压 ①～④（分四次运行，减轻单次压力）

**先运行上一格「解压共用设置」**。再按需依次运行下面四格；某一格失败时可单独重跑该格（可先删掉 `unzip/` 下对应不完整子目录）。


In [ ]:
# 解压 ① rsna-2022-cervical-spine-fracture-detection.zip -> unzip/rsna-2022-cervical-spine-fracture-detection/
run_one_zip("rsna-2022-cervical-spine-fracture-detection.zip", title="① 主包")


In [ ]:
# 解压 ② train_part_1.zip -> unzip/train_part_1/
run_one_zip("train_part_1.zip", title="② train_part_1")


In [ ]:
# 解压 ③ train_part_2.zip -> unzip/train_part_2/
run_one_zip("train_part_2.zip", title="③ train_part_2")


In [ ]:
# 解压 ④ train_part_3.zip -> unzip/train_part_3/
run_one_zip("train_part_3.zip", title="④ train_part_3")


### 解压完整性校验（文件数量）

**须先运行「解压共用设置」格**（以载入 `verify_extracted`）。下面 **四个代码格** 分别对照：**zip 内非目录文件条目数** vs **`unzip/` 下对应子目录内递归文件数**（跳过 `__MACOSX`）。二者相等且大于 0 视为通过。


In [ ]:
# 校验 ① 主包：rsna-2022-cervical-spine-fracture-detection.zip
verify_extracted("rsna-2022-cervical-spine-fracture-detection.zip", label="① 主包")


In [ ]:
# 校验 ② train_part_1.zip
verify_extracted("train_part_1.zip", label="② train_part_1")


In [ ]:
# 校验 ③ train_part_2.zip
verify_extracted("train_part_2.zip", label="③ train_part_2")


In [ ]:
# 校验 ④ train_part_3.zip
verify_extracted("train_part_3.zip", label="④ train_part_3")


In [ ]:
!pip install -q nibabel monai SimpleITK tqdm

### 切 patch 前先跑：了解目录结构

下面一格会打印 **`DATA_ROOT`**（优先 `unzip/`，否则 `download/`）、**目录树（深度有限）**、**扩展名统计**、**NIfTI 样例路径**、**含 `.dcm` 的叶子目录样例及每层切片数**。把输出贴到对话里，便于判断是否与当前「递归 NIfTI + 叶子 DICOM 序列」逻辑一致。


In [ ]:
# 切 patch 之前：探查 DATA_ROOT 下的文件结构（输出给人 / 助手读即可）
import os
from collections import Counter

DRIVE_DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/download"
DRIVE_UNZIP = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"


def get_volume_data_root():
    if os.path.isdir(DRIVE_UNZIP):
        for _, _, files in os.walk(DRIVE_UNZIP):
            if files:
                return DRIVE_UNZIP
    return DRIVE_DOWNLOAD


DATA_ROOT = get_volume_data_root()
SKIP = "__MACOSX"

print("=" * 72)
print("DATA_ROOT =", DATA_ROOT)
print("说明: 与后面切 patch 使用的根目录一致（unzip 非空则优先 unzip）。")
print("=" * 72)

if not os.path.isdir(DATA_ROOT):
    print("ERROR: 目录不存在，请先挂载 Drive 并确认路径。")
else:
    # --- 顶层一览 ---
    top = sorted(os.listdir(DATA_ROOT))
    print(f"\n[顶层] 共 {len(top)} 项（最多列 30 个）:")
    for name in top[:30]:
        p = os.path.join(DATA_ROOT, name)
        kind = "DIR " if os.path.isdir(p) else "FILE"
        print(f"  {kind}  {name}")
    if len(top) > 30:
        print(f"  ... 另有 {len(top) - 30} 项")

    # --- 扩展名统计（全树）---
    ext_cnt = Counter()
    n_files = 0
    n_dirs = 0
    max_depth = 0
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        n_dirs += len(dirnames)
        rel = os.path.relpath(dirpath, DATA_ROOT)
        depth = 0 if rel in (".", "") else rel.count(os.sep) + 1
        max_depth = max(max_depth, depth)
        for fn in filenames:
            n_files += 1
            lower = fn.lower()
            if "." in lower:
                ext_cnt[lower.rsplit(".", 1)[-1]] += 1
            else:
                ext_cnt["<no_ext>"] += 1

    print(f"\n[统计] 递归: 约 {n_dirs} 个子目录名出现、{n_files} 个文件；相对 DATA_ROOT 最大深度约 {max_depth}")
    print("[扩展名] 出现次数（降序，前 25）:")
    for ext, c in ext_cnt.most_common(25):
        print(f"  .{ext}: {c}")

    # --- 有限深度树（缩进 + 每级条数上限）---
    MAX_DEPTH = 4
    MAX_PER_LEVEL = 12

    def print_limited_tree(root, depth=0):
        indent = "  " * depth
        if depth > MAX_DEPTH:
            return
        try:
            names = [n for n in sorted(os.listdir(root)) if SKIP not in n and not n.startswith(".")]
        except OSError as e:
            print(indent + f"<无法列出: {e}>")
            return
        dirs = [n for n in names if os.path.isdir(os.path.join(root, n))]
        files = [n for n in names if n not in dirs]
        for name in dirs[:MAX_PER_LEVEL]:
            print(indent + "[DIR]  " + name + "/")
            print_limited_tree(os.path.join(root, name), depth + 1)
        if len(dirs) > MAX_PER_LEVEL:
            print(indent + f"... 还有 {len(dirs) - MAX_PER_LEVEL} 个子目录未显示")
        for name in files[:MAX_PER_LEVEL]:
            print(indent + "[FILE] " + name)
        if len(files) > MAX_PER_LEVEL:
            print(indent + f"... 还有 {len(files) - MAX_PER_LEVEL} 个文件未显示")

    print(f"\n[目录树] 深度<={MAX_DEPTH}，每级最多 {MAX_PER_LEVEL} 个目录 + {MAX_PER_LEVEL} 个文件（跳过隐藏与 __MACOSX）:")
    print_limited_tree(DATA_ROOT, 0)

    # --- NIfTI 路径样例 ---
    nii_samples = []
    for dirpath, _, files in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                nii_samples.append(os.path.relpath(os.path.join(dirpath, f), DATA_ROOT).replace("\\", "/"))
    nii_samples.sort()
    print(f"\n[NIfTI] 共 {len(nii_samples)} 个；前 15 条相对路径:")
    for p in nii_samples[:15]:
        print(" ", p)

    # --- DICOM：叶子目录（与预处理逻辑一致：有 .dcm 且无子目录）---
    leaf_dcm = []
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if not dcms:
            continue
        if dirnames:
            continue
        leaf_dcm.append((dirpath, len(dcms)))

    leaf_dcm.sort(key=lambda x: x[0])
    print(f"\n[DICOM 叶子目录] 共 {len(leaf_dcm)} 个（无子目录、内含 .dcm）；与预处理将逐目录读成一个序列。")
    print("  前 12 条: 相对路径 | 该目录 .dcm 数")
    for dirpath, k in leaf_dcm[:12]:
        rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
        print(f"  {k:5d}  {rel}")

    # --- 含 dcm 但仍有子目录的目录（预处理不会当整序列）---
    mixed = []
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if dcms and dirnames:
            mixed.append((dirpath, len(dcms), len(dirnames)))

    if mixed:
        print(f"\n[注意] {len(mixed)} 个目录「既有 .dcm 又有子目录」——当前脚本不会把它们当作单个 DICOM 序列；若你的数据在这里，需要改收集逻辑。")
        for dirpath, nd, ns in mixed[:8]:
            rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
            print(f"  {nd} dcm, {ns} subdirs  {rel}")

    # --- 可选：SimpleITK 试读前 2 个叶子序列的体素形状 ---
    try:
        import SimpleITK as sitk
    except ImportError:
        print("\n[试读] 未安装 SimpleITK，跳过体素形状探测（可先运行 pip 安装格）。")
    else:
        print("\n[试读 SimpleITK] 前 2 个叶子序列的 GetArrayFromImage().shape (z,y,x):")
        for dirpath, _ in leaf_dcm[:2]:
            try:
                r = sitk.ImageSeriesReader()
                names = r.GetGDCMSeriesFileNames(dirpath)
                r.SetFileNames(names)
                itk = r.Execute()
                arr = sitk.GetArrayFromImage(itk)
                rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
                print(f"  {rel}")
                print(f"    shape={tuple(arr.shape)}  slices={len(names)}")
            except Exception as e:
                rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
                print(f"  {rel}  -> 失败: {e}")

    print("\n" + "=" * 72)
    print("结构探查结束。确认无误后再运行「切 npy」那一格。")
    print("=" * 72)


In [ ]:
# RSNA-2022-CSFD：NIfTI + DICOM 序列，与 proc_spatial_sequence / AbdomenCT-1K Colab 一致
import os
import random
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

DRIVE_DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/download"
DRIVE_UNZIP = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"
DRIVE_SAVE_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy"


def get_volume_data_root():
    """优先 unzip；若为空则使用 download（仅解压到 download 时）。"""
    if os.path.isdir(DRIVE_UNZIP):
        for _, _, files in os.walk(DRIVE_UNZIP):
            if files:
                return DRIVE_UNZIP
    return DRIVE_DOWNLOAD


DATA_ROOT = get_volume_data_root()
print(f"体积扫描根目录 DATA_ROOT = {DATA_ROOT}")

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0

# 若只想用训练侧 CT 做预训练，设为 False（排除主包内 test_images/）
RSNA_INCLUDE_TEST_IMAGES = True


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode="trilinear"),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_zyx(image_zyx, patch_size_list, patch_num, save_root, start_index, tar_img_size, ds_name, vol_id):
    """image_zyx: (z,y,x)，与 cut_patch / MONAI 链一致。"""
    if len(image_zyx.shape) == 4:
        return []
    image = image_zyx
    n, patch_path_list = 0, []
    image_index = 0
    _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
    for i in range(_patch_num):
        n += 1
        patch_size = random.choice(patch_size_list)
        image_patch, _ = cut_patch(image, patch_size)
        image_patch = image_patch.transpose((2, 1, 0))
        image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
        save_name = os.path.join(save_root, "%s_%s_%s_%d.nii.gz" % (ds_name, vol_id, image_index, start_index + n))
        patch_path_list.append(save_name)
        np.save(save_name.replace(".nii.gz", ".npy"), image_patch)
    return patch_path_list


def cut_and_save_one_nii(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, _affine = load_nii_data(image_file)
    path_parts = image_file.replace("\\", "/").split("/")
    ds_name = path_parts[-3] if len(path_parts) >= 3 else "RSNA-CSFD"
    nii_id = path_parts[-1].split(".nii.gz")[0].split(".nii")[0].split(".mha")[0]
    if len(image.shape) == 4:
        return []
    image = image.transpose((2, 1, 0))
    return cut_and_save_one_volume_zyx(image, patch_size_list, patch_num, save_root, start_index, tar_img_size, ds_name, nii_id)


def load_dicom_series_zyx(series_dir):
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(series_dir)
    if not names:
        return None
    reader.SetFileNames(names)
    itk = reader.Execute()
    arr = sitk.GetArrayFromImage(itk)
    return arr.astype(np.float32)


def collect_nii_files(root):
    """收集 nii；跳过 segmentations/（RSNA 分割标注，非 CT 体积）。"""
    out = []
    if not os.path.isdir(root):
        return out
    for dirpath, _, files in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if "segmentations" in norm.split("/"):
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


def collect_dicom_series_dirs(root):
    """叶子目录、仅含 .dcm、无子目录 — 对应 StudyUID 单序列文件夹。"""
    series_dirs = []
    if not os.path.isdir(root):
        return series_dirs
    for dirpath, dirnames, filenames in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if not RSNA_INCLUDE_TEST_IMAGES and "test_images" in norm.split("/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if not dcms:
            continue
        if dirnames:
            continue
        series_dirs.append(dirpath)
    return sorted(series_dirs)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
nii_list = collect_nii_files(DATA_ROOT)
dicom_list = collect_dicom_series_dirs(DATA_ROOT)
print(f"NIfTI: {len(nii_list)}  |  DICOM 序列目录: {len(dicom_list)}")

if len(nii_list) == 0 and len(dicom_list) == 0:
    print("未找到 .nii/.nii.gz 或 DICOM 序列，请检查 DATA_ROOT 下是否已有竞赛数据。")
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(nii_list, desc="NIfTI")):
        patch_list = cut_and_save_one_nii(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 50 == 0:
            print(f"[{i+1}/{len(nii_list)}] {path} -> {len(patch_list)} npy")

    for i, series_dir in enumerate(tqdm(dicom_list, desc="DICOM")):
        try:
            vol = load_dicom_series_zyx(series_dir)
        except Exception as e:
            print(f"跳过（读取失败）: {series_dir}\n  {e}")
            continue
        if vol is None:
            continue
        rel = os.path.relpath(series_dir, DATA_ROOT).replace("\\", "/")
        vol_id = rel.replace("/", "_")
        ds_name = "RSNA-CSFD"
        patch_list = cut_and_save_one_volume_zyx(vol, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE, ds_name, vol_id)
        patch_list_all += patch_list
        if (i + 1) % 100 == 0:
            print(f"[{i+1}/{len(dicom_list)}] {rel} -> {len(patch_list)} npy")

    print(f"\n完成! 共 {len(patch_list_all)} 个 npy, 输出: {DRIVE_SAVE_ROOT}")

In [ ]:
# （可选）生成 train_patch_spatial.txt 供 pretrain 使用
import os

DRIVE_SAVE_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy"
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), "pretrain_lists")
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith(".npy")])
with open(os.path.join(list_save_dir, "train_patch_spatial.txt"), "w") as f:
    f.write("\n".join(npy_files))
print(f"train_patch_spatial.txt 已保存: {list_save_dir}（共 {len(npy_files)} 行）")